In [0]:
from pyspark.sql import functions as F

df = spark.table("bronze.sop_forecast.vendas_raw")

df_silver = (
    df.dropDuplicates(["date", "store", "item"])
      .withColumn("data", F.to_date("date"))
      .withColumn("loja_id", F.col("store").cast("int"))
      .withColumn("item_id", F.col("item").cast("int"))
      .withColumn("vendas", F.col("sales").cast("int"))
      .withColumn("ano_mes", F.date_trunc("month", F.col("data")))
      .filter(F.col("vendas") >= 0)
      .select("data", "loja_id", "item_id", "vendas", "ano_mes")
)

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.sop_forecast.vendas_limpo"))

print(f"Linhas Bronze : {df.count():,}")
print(f"Linhas Silver : {df_silver.count():,}")